In [1]:
import xmlrpc.client

# =========================================================================
# 1. ENVIRONMENT INITIALIZATION & PROXY CONNECTION
# =========================================================================
URL = "http://localhost:8069"
DB = "m.nushath"       
USERNAME = "user@company.com"  
API_KEY = "user"              

print("Initializing XML-RPC connections for Update Demo...")
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})
if not uid:
    print("[CRITICAL] Authentication failed!")
    exit()

print(f"[SUCCESS] Connected securely as User ID: {uid}\n" + "="*70)

# Fetch context parameters
user_data = models.execute_kw(DB, uid, API_KEY, 'res.users', 'read', [[uid]], {'fields': ['company_id', 'company_ids']})
current_company_id = user_data[0]['company_id'][0]
allowed_company_ids = user_data[0]['company_ids']
base_context = {'company_id': current_company_id, 'allowed_company_ids': allowed_company_ids}

Initializing XML-RPC connections for Update Demo...
[SUCCESS] Connected securely as User ID: 5


In [3]:
# =========================================================================
# DYNAMIC TARGET RESOLVER (Finds an existing order to update safely)
# =========================================================================
# We look for a draft order ('quotation') that has lines so we can modify them
order_search = models.execute_kw(DB, uid, API_KEY, 'sale.order', 'search_read',
    [[('state', '=', 'draft'), ('company_id', '=', current_company_id)]],
    {'fields': ['id', 'name', 'order_line'], 'limit': 1, 'context': base_context}
)

if not order_search:
    print("[ERROR] Setup Incomplete: Please run your Create script first to ensure a draft Sales Order exists to update.")
    exit()

target_order = order_search[0]
target_order_id = target_order['id']
target_order_lines = target_order['order_line']

print(f"Targeting Sales Order: '{target_order['name']}' (ID: {target_order_id})")
print(f"Current Sub-Line IDs attached to this order: {target_order_lines}")

# Pull a product variant ID to use for additions
product_search = models.execute_kw(DB, uid, API_KEY, 'product.product', 'search', [[['sale_ok', '=', True]]], {'limit': 1})
target_product_id = product_search[0] if product_search else False

Targeting Sales Order: 'S00060' (ID: 59)
Current Sub-Line IDs attached to this order: [93, 94]


In [4]:
# =========================================================================
# DEMO 1: STANDARD FIELD OVERWRITES & MANY2ONE CHANGES
# Documentation Link: Recipe #1
# =========================================================================
print("\n[DEMO 1] Updating Standard Text Columns and Many2one References...")

update_header_vals = {
    'client_order_ref': 'PO-SYNC-2026-REV1',
    'note': 'Header updated via XML-RPC configuration documentation tool script.',
}

# Parameters positional structure: [[id_list], values_dict]
success = models.execute_kw(DB, uid, API_KEY, 'sale.order', 'write',
    [[target_order_id], update_header_vals],
    {'context': base_context}
)
print(f" -> Method output data type: {type(success)} (Natively returns True/False)")
print(f" -> [SUCCESS] Header updated: {success}")


[DEMO 1] Updating Standard Text Columns and Many2one References...
 -> Method output data type: <class 'bool'> (Natively returns True/False)
 -> [SUCCESS] Header updated: True


In [5]:
# =========================================================================
# DEMO 2: COMPLEX ONE2MANY LINES MUTATION (ADD, UPDATE, DELETE)
# Documentation Link: Recipe #2 - Triplet Sequence Command List
# =========================================================================
print("\n" + "-"*50 + "\n[DEMO 2] Processing Nested Line Items Mutations...")

if len(target_order_lines) < 1:
    print(" -> Skipped: Target order has no lines to demonstrate line changes on.")
else:
    # Let's isolate the first sub-line item ID to update or delete
    existing_line_id = target_order_lines[0]
    
    update_lines_vals = {
        'order_line': [
            # COMMAND (1, ID, {vals}): Update the quantity of our first existing sub-line
            (1, existing_line_id, {
                'product_uom_qty': 99.0,
                'name': '[UPDATED VIA API] Optimized Quantity Adjustment'
            }),
            
            # COMMAND (0, 0, {vals}): Append a brand new line to this existing order
            (0, 0, {
                'product_id': target_product_id,
                'product_uom_qty': 3.0,
                'price_unit': 450.00,
                'name': '[NEW LINE VIA API] Added during write operation'
            })
        ]
    }
    
    # If the order has multiple lines, let's hard-delete the second line item using (2, ID, 0)
    if len(target_order_lines) > 1:
        line_to_delete = target_order_lines[1]
        print(f" -> Injecting delete request command for secondary line ID: {line_to_delete}")
        update_lines_vals['order_line'].append((2, line_to_delete, 0))

    # Fire the single atomic update command list
    line_success = models.execute_kw(DB, uid, API_KEY, 'sale.order', 'write',
        [[target_order_id], update_lines_vals],
        {'context': base_context}
    )
    print(f" -> [SUCCESS] Relational line command mutation execution state: {line_success}")


--------------------------------------------------
[DEMO 2] Processing Nested Line Items Mutations...
 -> Injecting delete request command for secondary line ID: 94
 -> [SUCCESS] Relational line command mutation execution state: True


In [8]:
# =========================================================================
# DEMO 3: MASS CLEANING RELATIONAL LISTS (COMMAND 5)
# Documentation Link: Recipe #2 - (5, 0, 0) Clear All
# =========================================================================
print("\n" + "-"*50 + "\n[DEMO 3] Demonstrating Mass Deletion / Wiping Out Sub-Lines...")

# WARNING: Running this will clear out all line objects currently attached to the record.
# To keep your test data intact, this demo fires on a conditional mock loop flag.
execute_wipe_out = False 

if not execute_wipe_out:
    print(" -> Skipped: Set 'execute_wipe_out = True' inside the script to test clearing all lines.")
else:
    wipe_payload = {
        'order_line': [(5, 0, 0)] # Drops every single associated item line row
    }
    wipe_success = models.execute_kw(DB, uid, API_KEY, 'sale.order', 'write',
        [[target_order_id], wipe_payload],
        {'context': base_context}
    )
    print(f" -> [SUCCESS] All sub-lines cleared out completely: {wipe_success}")

print("\n" + "="*70 + "\n[FINISHED] Complete write/update orchestration demonstration completed.")


--------------------------------------------------
[DEMO 3] Demonstrating Mass Deletion / Wiping Out Sub-Lines...
 -> Skipped: Set 'execute_wipe_out = True' inside the script to test clearing all lines.

[FINISHED] Complete write/update orchestration demonstration completed.
